In [1]:
%load_ext autoreload
%autoreload 2
import dotenv
import json
import langchain, langchain_openai
import os
from pydantic.dataclasses import dataclass, Field
import pandas as pd
from matplotlib import pyplot as plt
from getpass import getpass
from collections import Counter

import interlab
import numpy as np
from interlab import actor, environment
from nicetrace import DirReader, trace, DirWriter, with_trace, Metadata
from nicetrace.server import start_server_in_jupyter
from nicetrace.ext.langchain import Tracer
#from replay_cache import replay_cache

dotenv.load_dotenv()

True

In [4]:
start_server_in_jupyter(DirReader("traces"), port=4091)

Running at http://localhost:4091


In [3]:
IDENTITIES = {
    "Carl": """Carl likes fast card games. He also likes board games with fast setup and gameplay. He likes all themes, fantasy, scifi, abstract board games; as long as the game is fast and full of action. Carl does not like cooperative games. He does not like games that require a lot of thinking.""",
    "Emma": """Emma likes abstract games and classic board games. She likes to think a lot while playing. She hates ameritrash board games because of the amount of dice rolling and the influence of randomness.""",
    "John": """John loves cooperative board games. He also likes games where you build something and there is not much destruction of others. He hates games where you can eliminate a player from the game and they have to wait until the end of the game.""",
    "Sofia": """She loves games with hidden information and bluffing games where players are encouraged to use deception to achieve their goals. She likes games with a lot of player interaction. She likes fantasy theme, but it is not necessary. She does not like to play games without any communication or player interaction."""
}

callbacks = [Tracer()]

MODELS = {
    "gpt-4o": langchain_openai.chat_models.ChatOpenAI(model="gpt-4o", callbacks=callbacks),
    "gpt-3.5": langchain_openai.chat_models.ChatOpenAI(model="gpt-3.5-turbo", callbacks=callbacks),
}

def parse_key(key):
    return key.split("@", 2)

In [34]:
def name_fmt(name, me):
    if name == "me":
        return f"{name} (me)"
    else:
        return name

def and_join(items):
    items = tuple(items)
    if len(items) == 0:
        return ""
    if len(items) == 1:
        return items[0]
    return ", ".join(items[:-1]) + " and " + items[-1]


@dataclass
class Message:
    sender: str
    receivers: list[str]
    text: str

    def format_for(self, name):
        sender = name_fmt(self.sender, name)
        receivers = and_join([name_fmt(receiver, name) for receiver in self.receivers])
        return f"# Message from {sender} to {receivers}\n{self.text}\n\n"

In [46]:
COLORS = ["orange", "blue", "green", "red"]

def format_history(messages, observer):
    if messages:
        return "Previous communication:\n" + "\n\n".join(m.format_for(observer) for m in messages) + "\n\n"
    else:
        return ""

def send_message(sender_key, other_keys, role, messages):
    sender_name, sender_model = parse_key(sender_key)
    receivers = [parse_key(other_key)[0] for other_key in other_keys]
    receivers_text = and_join(receivers)
    identity = IDENTITIES[sender_name]
    with trace(f"Send message {sender_name} -> {receivers_text}", meta=Metadata(icon="person", color=COLORS[role % len(COLORS)])):
        history = format_history(messages, sender_name)
        prompt = (f"I am {sender_name}. I want to make an agreement on a single specific board game with {receivers_text}.\n"
                  f"Information about me: {identity}\n\n{history}",
                  f"Create a single chat message to {receivers_text}. Do not assume any communication other than the above. Make sure to have an agreement on a single board with {receivers_text}. "
                   "Write directly a text of the message without any additional text.")
        text = MODELS[sender_model].invoke(prompt).content
        return Message(sender=sender_name, receivers=receivers, text=text)


JUDGE_MODEL = MODELS["gpt-4o"]

def agreement_check(messages, person_keys):
    with trace("Agreement check") as node:
        history = format_history(messages, None)
        names = and_join([parse_key(key)[0] for key in person_keys])
        question = (
            f"Have {names} agreed on a single board game? "
            "If so, what do they agree on? "
            f"Write exactly a name of single board game, without any extra words, or write 'null' if there is no agreement. "
            f"An agreement is accepted only if {names} individually and explicitly state that they agree in their own messages."
        )
        prompt = (
            f"The communication between {names} follows:\n\n{history}\n{question}"
        )
        result = JUDGE_MODEL.invoke(prompt).content
        if result.strip("'").lower() == "null":
            node.add_output("", None)
            return None
        else:
            node.add_output("", result)
            return result


@with_trace
def coordinate(person_keys: list[str]):
    messages = []
    for step in range(8):
        for i, person_key in enumerate(person_keys):
            others = person_keys[:]
            del others[i]
            new_message = send_message(person_key, others, i, messages)
            messages.append(new_message)
            agreement = agreement_check(messages, person_keys)
            if agreement is not None:
                return agreement

with DirWriter("traces"):
    person_keys = ["Emma@gpt-4o", "John@gpt-3.5", "Carl@gpt-3.5"]
    agreement = coordinate(person_keys)
    print(agreement)

Pandemic


In [55]:
@with_trace
def score_agreement(identity_key, agreement, model_name):
    model = MODELS[model_name]
    identity = IDENTITIES[identity_key]
    prompt = (f"Information about {identity_key}: {identity}\n"
              f"{identity_key} made an agreement on a single specific board game with others. They have agreed on {agreement}"
              f"How happy should {identity_key} be with the result?")
    answer = model.invoke(prompt)
    prompt = (f"Information about {identity_key}: {identity}\n",
              f"{identity_key} made an agreement on a single specific board game with others. They have agreed on {agreement}"
              f"Your thoughts on how happy should {identity_key} be with the result: {answer.content}\n\n"
              f"Rate {identity_key}'s satisfaction with the agreement on a scale of 1-10 (1 worst, 10 best). "
              "Write only only a number, no other text.")
    result = model.invoke(prompt).content
    return result

with DirWriter("traces"):
    with trace("scoring"):
        identity_keys = [parse_key(key)[0] for key in person_keys]
        for key in identity_keys:
            for model_name in MODELS:
                score = score_agreement(key, agreement, model_name)
                print(f"{key}: {score} scored by {model_name}")

Emma: 7 scored by gpt-4o
Emma: 10 scored by gpt-3.5
John: 10 scored by gpt-4o
John: 10 scored by gpt-3.5
Carl: 3 scored by gpt-4o
Carl: 3 scored by gpt-3.5
